In [ ]:
# ============================================================
# Proyecto Integrador M4 – IA Generativa con RAG + CSV + PDF único
# ============================================================

!pip install -q --upgrade --no-cache-dir "langchain>=1.0" langchain-openai langchain-elasticsearch langchain-community langchain-text-splitters pypdf elasticsearch pandas fpdf2 psycopg[binary,pool]==3.2.6 langgraph

import os, datetime, pandas as pd
from pathlib import Path
from fpdf import FPDF

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.messages import HumanMessage, SystemMessage
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse, before_agent

from psycopg_pool import ConnectionPool


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 266.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 141.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 366.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 291.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 198.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 214.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 370.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 318.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 193.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 379.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ------------------------------------------------------------
# 1. Configuración de credenciales
# ------------------------------------------------------------
openai_key = os.getenv("OPENAI_API_KEY") or open("/content/api_key.txt").read().strip()
os.environ["OPENAI_API_KEY"] = openai_key

ES_URL = os.getenv("ES_URL", "http://34.125.210.205:9200")
ES_USER = os.getenv("ES_USER", "elastic")
ES_PASSWORD = os.getenv("ES_PASSWORD") or open("/content/elasticstore_urp.txt").read().strip()
INDEX_NAME = os.getenv("ES_INDEX", "rag_fallas_mantenimiento_v1")

# ------------------------------------------------------------
# 2. Cargar datos simulados (CSV)
# ------------------------------------------------------------
equipos = pd.read_csv("/content/equipos.csv", encoding='latin1', sep=';', header=0)
repuestos = pd.read_csv("/content/repuestos.csv", encoding='latin1', sep=';', header=0)


In [ ]:
# ------------------------------------------------------------
# 3. RAG – Ingesta (carga de PDF único)
# ------------------------------------------------------------
from langchain_community.document_loaders import PyPDFLoader
pdf_path = "/content/historial_fallas_SAP.pdf"  # Documento único con todas las fallas
loader = PyPDFLoader(pdf_path)
pages = loader.load()
print(f"Páginas cargadas: {len(pages)}")
print("Metadata base:", pages[0].metadata)

/tmp/ipykernel_747/3752688358.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Páginas cargadas: 4
Metadata base: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-06-14T21:07:24-05:00', 'author': 'CORNELIO Miguel       TGP', 'moddate': '2026-06-14T21:07:24-05:00', 'source': '/content/historial_fallas_SAP.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}


In [ ]:
# ------------------------------------------------------------
# 4. RAG – Enriquecimiento de metadata
# ------------------------------------------------------------
from langchain_core.documents import Document

def enrich_page_document(doc: Document) -> Document:
    """
    Enriquecer un Document con metadata adicional para RAG.
    Se asume que el contenido del doc incluye información de falla y solución.
    """
    # Extraer datos básicos del contenido
    contenido = doc.page_content.lower()

    # Detectar tipo de falla por palabras clave
    if "arranque" in contenido:
        tipo_falla = "arranque"
    elif "tensión" in contenido or "voltaje" in contenido:
        tipo_falla = "tension"
    elif "sobrecalentamiento" in contenido:
        tipo_falla = "sobrecalentamiento"
    elif "vibración" in contenido:
        tipo_falla = "vibracion"
    elif "presión" in contenido or "caudal" in contenido:
        tipo_falla = "presion"
    elif "fuga" in contenido:
        tipo_falla = "fuga"
    else:
        tipo_falla = "otro"

    # Crear metadata enriquecida
    enriched_metadata = {
        "equipo_id": doc.metadata.get("equipo_id", "No definido"),
        "nombre_equipo": doc.metadata.get("nombre_equipo", "No definido"),
        "ubicacion": doc.metadata.get("ubicacion", "No definida"),
        "criticidad": doc.metadata.get("criticidad", "No definida"),
        "fecha": doc.metadata.get("fecha", "No definida"),
        "tipo_falla": tipo_falla,
        "solucion": doc.metadata.get("solucion", "No definida")
    }

    # Retornar nuevo Document con metadata enriquecida
    return Document(page_content=doc.page_content, metadata=enriched_metadata)

enriched_docs = [enrich_page_document(doc) for doc in pages]

print("Documento enriquecido de ejemplo:")
print(enriched_docs[1].metadata)

Documento enriquecido de ejemplo:
{'equipo_id': 'No definido', 'nombre_equipo': 'No definido', 'ubicacion': 'No definida', 'criticidad': 'No definida', 'fecha': 'No definida', 'tipo_falla': 'arranque', 'solucion': 'No definida'}


In [ ]:
# ------------------------------------------------------------
# 5. RAG – Fragmentación
# ------------------------------------------------------------
# La mejor herramientas para la fragmentación de el PDF "Historial de fallas SAP" es SemanticChunker, porque cada chunk representa una falla completa y evita confusiones en la recuperación.
# No obstante, es más costoso en costo y embedding. Por ello, por la naturaleza del PDF es más óptimo usar RecursiveCharacterTextSplitter, dado que es más simpleo y rápido.

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = splitter.split_documents(enriched_docs)

print(f"Chunks generados: {len(splits)}")
print("Chunk de ejemplo:")
print(splits[0].metadata)

Chunks generados: 13
Chunk de ejemplo:
{'equipo_id': 'No definido', 'nombre_equipo': 'No definido', 'ubicacion': 'No definida', 'criticidad': 'No definida', 'fecha': 'No definida', 'tipo_falla': 'arranque', 'solucion': 'No definida'}


In [ ]:
# ------------------------------------------------------------
# 6. RAG – Embeddings y Almacenamiento (Indexación en Elasticsearch)
# ------------------------------------------------------------
from langchain_openai import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

vector_store = ElasticsearchStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    es_url=ES_URL,
    es_user=ES_USER,
    es_password=ES_PASSWORD,
)

if vector_store.client.indices.exists(index=INDEX_NAME):
    vector_store.client.indices.delete(index=INDEX_NAME)

ids = [f"chunk_{i}" for i in range(len(splits))]
vector_store.add_documents(documents=splits, ids=ids)
vector_store.client.indices.refresh(index=INDEX_NAME)

inserted_ids = vector_store.add_documents(
    documents=splits,
    ids=ids,
    bulk_kwargs={"chunk_size": 100},
)

vector_store.client.indices.refresh(index=INDEX_NAME)
print(f"Chunks indexados: {len(inserted_ids)}")

#Convierte cada chunk de texto en un vector numérico usando el modelo text-embedding-3-large. Estos vectores capturan el significado semántico del texto.
#El almacenamiento con Elasticsearch permite combinar búsquedas vectoriales con filtros tradicionales (ej. por fecha o equipo).


Chunks indexados: 13


In [ ]:
# ------------------------------------------------------------
# 8. Herramientas del agente
# ------------------------------------------------------------
@tool
def procesar_reporte(texto: str) -> dict:
    """Procesa un texto para encontrar un ID de equipo y devuelve la descripción."""
    for eid in equipos["ID"].astype(str):
        if eid in texto:
            return {"id_equipo": eid, "descripcion": texto, "reporte": texto}
    return {"error": "ID no encontrado"}

@tool
def integrar_datos(id_equipo: str) -> dict:
    """Integra datos de equipos y repuestos para un ID de equipo dado."""
    eq = equipos[equipos["ID"] == id_equipo].iloc[0]
    rep = repuestos[repuestos["ID_Equipo"] == id_equipo].iloc[0]
    return {
        "id_equipo": id_equipo,
        "nombre": eq["Nombre"],
        "ubicacion": eq["Ubicacion"],
        "criticidad": eq["Criticidad"],
        "repuesto": rep["Repuesto"],
        "stock": rep["Stock"],
        "tiempo_reposicion": rep["TiempoReposicion"]
    }

@tool
def consultar_historial(query: str) -> list:
    """Consulta el historial de fallas y soluciones usando similitud semántica."""
    results = vector_store.similarity_search_with_score(query, k=2)
    salida = []
    for doc, score in results:
        salida.append(
            {
                "contenido": doc.page_content,
                "fuente": doc.metadata.get("source_file", "PDF externo"),
                "pagina": doc.metadata.get("page_number", "N/A"),
                "score": score,
            }
        )
    return salida

from fpdf import FPDF

orden_counter = 1  # contador global para numerar las órdenes

@tool
def generar_orden_trabajo(datos: dict) -> str:
    """Genera una orden de trabajo en PDF con formato estructurado."""
    global orden_counter

    # Crear PDF
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)

    # Título con número y prioridad
    prioridad = datos.get("criticidad", "No definida")
    titulo = f"Orden de Trabajo n° {orden_counter}  Prioridad {prioridad}"
    pdf.cell(200, 10, titulo, ln=True, align="C")
    pdf.ln(10)

    # Historial
    historial = datos.get('historial_fallas', [])
    if isinstance(historial, list):
        historial_texto = "\n".join([f"- {h}" for h in historial])
    else:
        historial_texto = historial

    # Contenido estructurado
    pdf.multi_cell(
        0,
        10,
        f"""
1. Falla reportada: {datos.get('reporte')}
2. Fecha y hora de la falla reportada: {datos.get("fecha_reporte", datetime.datetime.now().strftime("%d/%m/%Y %H:%M"))}
3. Ubicación: {datos.get('ubicacion', 'No especificada')}
4. ID Equipo: {datos.get('id_equipo', 'No especificado')}
5. Criticidad: {datos.get('criticidad', 'No definida')}
6. Nota preventiva: {datos.get('nota_preventiva', 'No definida')}
7. Historial de fallas:
{historial_texto}
8. Stock de repuesto: {datos.get('stock', '0')}, Tiempo de reposición: {datos.get('tiempo_reposicion', 'No definido')}
9. Responsable: {datos.get('responsable', 'Tecnico 2')}
""",
    )

    # Guardar PDF con nombre único
    nombre_pdf = f"/content/Orden_trabajo_{orden_counter}.pdf"
    pdf.output(nombre_pdf)

    # Incrementar contador para la siguiente orden
    orden_counter += 1

    return f"Orden de trabajo generada: {nombre_pdf}"


In [ ]:
# ------------------------------------------------------------
# 9. Middlewares
# ------------------------------------------------------------
@before_agent(can_jump_to=["end"])
def validar_reporte(state, runtime):
    texto_usuario = ""
    for msg in state["messages"]:
        if msg.type == "human":
            texto_usuario = msg.content
    if not any(eid in texto_usuario for eid in equipos["ID"].astype(str)):
        return {"messages": [SystemMessage(content="⚠️ Error: ID de equipo inválido.")], "jump_to": "end"}
    return None

class FallbackMiddleware(AgentMiddleware):
    def wrap_model_call(self, request: ModelRequest, handler: callable) -> ModelResponse:
        try:
            return handler(request)
        except Exception:
            mensaje_fallback = SystemMessage(content="⚠️ No se pudo procesar el reporte.")
            return ModelResponse(result=[mensaje_fallback])


In [ ]:
# ------------------------------------------------------------
# 10. Creación del agente híbrido con RAG + PDF
# ------------------------------------------------------------
modelo = init_chat_model("openai:gpt-4.1", temperature=0)

agente_mantenimiento = create_agent(
    model=modelo,
    tools=[
        procesar_reporte,       # Extrae ID y descripción
        integrar_datos,         # Consulta CSV de equipos y repuestos
        consultar_historial,    # Recupera evidencia del PDF indexado (RAG)
        generar_orden_trabajo   # Genera PDF estructurado con orden de trabajo
    ],
    middleware=[validar_reporte, FallbackMiddleware()],
    system_prompt="""
    Eres un agente de mantenimiento con RAG.
    - Valida ID y ubicación antes de procesar.
    - Consulta el nombre del equipo, ID y ubicación antes de procesar.
    - Consulta criticidad y stock de repuesto desde CSV.
    - Recupera todas las evidencia de fallas históricas desde la herramienta consultar_historial.
    - Resume el historial de fallas sólo de lo que están relacionado con la falla reportada, de manera ordenada con fechas y soluciones.
    - Colocar una Nota preventiva basada en las fallas históricas recuperadas.
    - Finalmente, Genera una orden de trabajo en PDF con formato estructurado (9 puntos), sin efectuar más preguntas, usando siempre:
        * 'Nombre' del CSV para el nombre real del equipo.
        * 'reporte' para mostrar la falla reportada.
        * 'responsable' para identificar al técnico, colocar de manera aleatoria los responsables con nombre genéricos.
        * Historial de fallas.
     - Cada ejecución debe producir un archivo único (Orden_trabajo_X.pdf).
    """
)


In [ ]:
# ============================================================
# Ejemplos de prueba – Proyecto M4
# ============================================================

# Caso 1 – Bomba EQA-1001 (Planta Cusco)
reporte1 = "La bomba EQA-1001 en Planta Cusco presenta baja presión y vibración excesiva."
config1 = {"configurable": {"thread_id": "prueba_bomba_cusco"}}

resultado1 = agente_mantenimiento.invoke({"messages": [HumanMessage(content=reporte1)]}, config=config1)
print("🔹 Resultado Caso 1:")
for msg in resultado1["messages"]:
    print(msg.content)

/tmp/ipykernel_747/4247685488.py:55: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", size=12)
/tmp/ipykernel_747/4247685488.py:60: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(200, 10, titulo, ln=True, align="C")


🔹 Resultado Caso 1:
La bomba EQA-1001 en Planta Cusco presenta baja presión y vibración excesiva.

{"id_equipo": "EQA-1001", "descripcion": "La bomba EQA-1001 en Planta Cusco presenta baja presión y vibración excesiva.", "reporte": "La bomba EQA-1001 en Planta Cusco presenta baja presión y vibración excesiva."}

{'id_equipo': 'EQA-1001', 'nombre': 'Bomba Principal', 'ubicacion': 'Planta-Cusco', 'criticidad': 'Alta', 'repuesto': 'Bomba Principal', 'stock': np.int64(0), 'tiempo_reposicion': '30 días'}

[{"contenido": "filtros. \n• 30/05/2025: Pérdida de presión → Solución: Ajuste de válvulas. \n• 12/09/2025: Vibración excesiva → Solución: Balanceo dinámico. \n• 20/11/2025: Fuga de aire → Solución: Cambio de empaquetadura. \nEQA-1004 – Generador Diesel | Planta Cusco | Criticidad: Alta \n• 05/03/2025: Fallo en arranque → Solución: Cambio de batería.", "fuente": "PDF externo", "pagina": "N/A", "score": 0.79920244}, {"contenido": "Empresa: Hidrocarburos del Perú \nDocumento: Historial de Fa

In [ ]:
# Caso Error de ID (no existe en CSV)
reporte2 = "La bomba EQA-9999 en Planta Cusco presenta fuga en el sello."
config2 = {"configurable": {"thread_id": "prueba_error_id"}}

resultado2 = agente_mantenimiento.invoke({"messages": [HumanMessage(content=reporte2)]}, config=config2)
print("\n🔹 Resultado Caso Error esperado:")
for msg in resultado2["messages"]:
    print(msg.content)


🔹 Resultado Caso Error esperado:
La bomba EQA-9999 en Planta Cusco presenta fuga en el sello.
⚠️ Error: ID de equipo inválido.


In [ ]:
# Caso 2 – Generador Eléctrico (Planta Lima)
reporte3 = "El generador EQA-2004 en Planta Lima no arranca y muestra caída de tensión."
config3 = {"configurable": {"thread_id": "prueba_generador_lima"}}

resultado3 = agente_mantenimiento.invoke({"messages": [HumanMessage(content=reporte3)]}, config=config3)
print("\n🔹 Resultado Caso 2:")
for msg in resultado3["messages"]:
    print(msg.content)

/tmp/ipykernel_747/4247685488.py:55: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", size=12)
/tmp/ipykernel_747/4247685488.py:60: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(200, 10, titulo, ln=True, align="C")



🔹 Resultado Caso 2:
El generador EQA-2004 en Planta Lima no arranca y muestra caída de tensión.

{"id_equipo": "EQA-2004", "descripcion": "El generador EQA-2004 en Planta Lima no arranca y muestra caída de tensión.", "reporte": "El generador EQA-2004 en Planta Lima no arranca y muestra caída de tensión."}

{'id_equipo': 'EQA-2004', 'nombre': 'Generador Eléctrico', 'ubicacion': 'Planta-Lima', 'criticidad': 'Media', 'repuesto': 'Generador Eléctrico', 'stock': np.int64(4), 'tiempo_reposicion': '5 días'}

[{"contenido": "EQA-4003 – Compresor D | Planta Piura | Criticidad: Alta \n• 16/02/2025: Sobrecalentamiento → Solución: Cambio de aceite. \n• 02/06/2025: Fuga de aire → Solución: Cambio de empaquetadura. \n• 20/09/2025: Vibración excesiva → Solución: Balanceo dinámico. \n• 28/11/2025: Pérdida de presión → Solución: Ajuste de válvulas. \nEQA-4004 – Generador Principal | Planta Piura | Criticidad: Baja \n• 09/03/2025: Fallo en arranque → Solución: Cambio de batería.", "fuente": "PDF extern

In [ ]:
# Caso 4 – Compresor (Planta Arequipa)
reporte4 = "El compresor EQA-3003 en Planta Arequipa presenta fuga de aire y sobrecalentamiento."
config4 = {"configurable": {"thread_id": "prueba_compresor_arequipa"}}

resultado4 = agente_mantenimiento.invoke({"messages": [HumanMessage(content=reporte4)]}, config=config4)
print("\n🔹 Resultado Caso 4:")
for msg in resultado4["messages"]:
    print(msg.content)

/tmp/ipykernel_747/4247685488.py:55: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", size=12)
/tmp/ipykernel_747/4247685488.py:60: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(200, 10, titulo, ln=True, align="C")



🔹 Resultado Caso 4:
El compresor EQA-3003 en Planta Arequipa presenta fuga de aire y sobrecalentamiento.

{"id_equipo": "EQA-3003", "descripcion": "El compresor EQA-3003 en Planta Arequipa presenta fuga de aire y sobrecalentamiento.", "reporte": "El compresor EQA-3003 en Planta Arequipa presenta fuga de aire y sobrecalentamiento."}

{'id_equipo': 'EQA-3003', 'nombre': 'Compresor C', 'ubicacion': 'Planta-Arequipa', 'criticidad': 'Alta', 'repuesto': 'Compresor C', 'stock': np.int64(1), 'tiempo_reposicion': '20 días'}
[{"contenido": "• 22/04/2025: Fuga en sello → Solución: Cambio de sello. \n• 05/07/2025: Pérdida de presión → Solución: Ajuste de calibración. \n• 19/10/2025: Vibración anómala → Solución: Reemplazo de soporte. \nEQA-3003 – Compresor C | Planta Arequipa | Criticidad: Alta \n• 14/02/2025: Sobrecalentamiento → Solución: Cambio de aceite. \n• 28/05/2025: Fuga de aire → Solución: Cambio de empaquetadura. \n• 11/09/2025: Vibración excesiva → Solución: Balanceo dinámico.", "fuent